# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import math
import os
import timeit

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

%matplotlib inline

device = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [2]:
def load_mnist_dataset():
    """
    Fetch the MNIST dataset from TensorFlow and perform preprocessing to prepare
    it for CNN classifiers.

    Total samples: 70,000. Default split: 50k train / 10k val / 10k test.
    Images: 28×28 grayscale, normalized and reshaped to (N, 28, 28, 1).
    """
    (x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

    X = np.concatenate([x_train_full, x_test], axis=0).astype(np.float32)
    y = np.concatenate([y_train_full, y_test], axis=0).astype(np.int32)
    X = np.expand_dims(X, axis=-1)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.2857, stratify=y, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )

    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)

    X_train = (X_train - mean_pixel) / (std_pixel + 1e-8)
    X_val = (X_val - mean_pixel) / (std_pixel + 1e-8)
    X_test = (X_test - mean_pixel) / (std_pixel + 1e-8)

    return X_train, y_train, X_val, y_val, X_test, y_test


# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_mnist_dataset()
print("Train data shape: ", X_train.shape)
print("Train labels shape: ", y_train.shape, y_train.dtype)
print("Validation data shape: ", X_val.shape)
print("Validation labels shape: ", y_val.shape)
print("Test data shape: ", X_test.shape)
print("Test labels shape: ", y_test.shape)

Train data shape:  (50001, 28, 28, 1)
Train labels shape:  (50001,) int32
Validation data shape:  (9999, 28, 28, 1)
Validation labels shape:  (9999,)
Test data shape:  (10000, 28, 28, 1)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False) -> None:
        """
        Construct a Dataset object to iterate over data X and labels y.

        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], "Got different numbers of data and labels"
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i : i + B], self.y[i : i + B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5:
        break

0 (64, 28, 28, 1) (64,)
1 (64, 28, 28, 1) (64,)
2 (64, 28, 28, 1) (64,)
3 (64, 28, 28, 1) (64,)
4 (64, 28, 28, 1) (64,)
5 (64, 28, 28, 1) (64,)
6 (64, 28, 28, 1) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(
            hidden_size,
            activation="relu",
            kernel_initializer=initializer,
        )
        self.fc2 = tf.keras.layers.Dense(
            num_classes,
            activation="softmax",
            kernel_initializer=initializer,
        )
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """A small unit test to exercise the TwoLayerFC model above."""
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)


test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.conv1 = tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=5,
            padding="same",
            kernel_initializer=initializer,
        )

        self.conv2 = tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=3,
            padding="same",
            kernel_initializer=initializer,
        )

        self.fc = tf.keras.layers.Dense(
            num_classes,
            kernel_initializer=initializer,
        )

        self.relu = tf.keras.layers.Activation("relu")
        self.softmax = tf.keras.layers.Activation("softmax")
        self.flatten = tf.keras.layers.Flatten()

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.relu(x)

        x = self.flatten(x)
        x = self.fc(x)
        scores = self.softmax(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores

In [7]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)


test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [8]:
def train_part34(
    model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False, print_every=100
):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name="train_loss")
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name="train_accuracy")

        val_loss = tf.keras.metrics.Mean(name="val_loss")
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name="val_accuracy")

        t = 0
        for epoch in range(num_epochs):
            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = "Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}"
                        print(
                            template.format(
                                t,
                                epoch + 1,
                                train_loss.result(),
                                train_accuracy.result() * 100,
                                val_loss.result(),
                                val_accuracy.result() * 100,
                            )
                        )
                    t += 1

In [9]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2


def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)


def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)


train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.9363784790039062, Accuracy: 9.375, Val Loss: 2.514397382736206, Val Accuracy: 20.332033157348633
Iteration 100, Epoch 1, Loss: 0.663776159286499, Accuracy: 79.0996322631836, Val Loss: 0.39807379245758057, Val Accuracy: 88.298828125
Iteration 200, Epoch 1, Loss: 0.5130159258842468, Accuracy: 84.2972640991211, Val Loss: 0.32717257738113403, Val Accuracy: 90.23902130126953
Iteration 300, Epoch 1, Loss: 0.44197842478752136, Accuracy: 86.67462158203125, Val Loss: 0.3031047582626343, Val Accuracy: 91.06910705566406
Iteration 400, Epoch 1, Loss: 0.405771940946579, Accuracy: 87.8428955078125, Val Loss: 0.27289965748786926, Val Accuracy: 91.88919067382812
Iteration 500, Epoch 1, Loss: 0.37742480635643005, Accuracy: 88.74750518798828, Val Loss: 0.2587098181247711, Val Accuracy: 92.15921783447266
Iteration 600, Epoch 1, Loss: 0.3563987612724304, Accuracy: 89.37187957763672, Val Loss: 0.25318795442581177, Val Accuracy: 92.41924285888672
Iteration 700, Epoch 1, Loss: 0

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [10]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10


def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model


def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate, momentum=0.9, nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer


train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.20259428024292, Accuracy: 12.5, Val Loss: 4.1471967697143555, Val Accuracy: 17.921791076660156
Iteration 100, Epoch 1, Loss: 0.5004536509513855, Accuracy: 86.09220123291016, Val Loss: 0.21228207647800446, Val Accuracy: 93.7593765258789
Iteration 200, Epoch 1, Loss: 0.33626222610473633, Accuracy: 90.58612823486328, Val Loss: 0.13901616632938385, Val Accuracy: 95.88958740234375
Iteration 300, Epoch 1, Loss: 0.2634127736091614, Accuracy: 92.58201599121094, Val Loss: 0.11687704920768738, Val Accuracy: 96.47964477539062
Iteration 400, Epoch 1, Loss: 0.22564925253391266, Accuracy: 93.57855224609375, Val Loss: 0.10600824654102325, Val Accuracy: 96.84968566894531
Iteration 500, Epoch 1, Loss: 0.20183065533638, Accuracy: 94.22093200683594, Val Loss: 0.09115136414766312, Val Accuracy: 97.35973358154297
Iteration 600, Epoch 1, Loss: 0.18405382335186005, Accuracy: 94.73533630371094, Val Loss: 0.08845826983451843, Val Accuracy: 97.33972930908203
Iteration 700, Epoch 1,

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [11]:
learning_rate = 1e-2


def model_init_fn():
    input_shape = (28, 28, 1)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(
            hidden_layer_size,
            activation="relu",
            kernel_initializer=initializer,
        ),
        tf.keras.layers.Dense(
            num_classes,
            activation="softmax",
            kernel_initializer=initializer,
        ),
    ]
    model = tf.keras.Sequential(layers)
    return model


def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)


train_part34(model_init_fn, optimizer_init_fn)

c:\Users\ddima\Desktop\SSAU\labs\TofAI\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 2.9971604347229004, Accuracy: 7.8125, Val Loss: 2.6179959774017334, Val Accuracy: 18.83188247680664
Iteration 100, Epoch 1, Loss: 0.6805557608604431, Accuracy: 78.72834014892578, Val Loss: 0.4029746651649475, Val Accuracy: 88.14881134033203
Iteration 200, Epoch 1, Loss: 0.5207428336143494, Accuracy: 84.13401794433594, Val Loss: 0.337955117225647, Val Accuracy: 89.8589859008789
Iteration 300, Epoch 1, Loss: 0.4477563500404358, Accuracy: 86.46698760986328, Val Loss: 0.30826514959335327, Val Accuracy: 90.97909545898438
Iteration 400, Epoch 1, Loss: 0.4108244478702545, Accuracy: 87.61689758300781, Val Loss: 0.2787007987499237, Val Accuracy: 91.62916564941406
Iteration 500, Epoch 1, Loss: 0.3823586404323578, Accuracy: 88.5853271484375, Val Loss: 0.26518967747688293, Val Accuracy: 92.00920104980469
Iteration 600, Epoch 1, Loss: 0.3598214387893677, Accuracy: 89.25228881835938, Val Loss: 0.26058176159858704, Val Accuracy: 92.17921447753906
Iteration 700, Epoch 1, Lo

Альтернативный менее гибкий способ обучения:

In [12]:
model = model_init_fn()
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=[tf.keras.metrics.sparse_categorical_accuracy],
)
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.3243 - sparse_categorical_accuracy: 0.9038 - val_loss: 0.3318 - val_sparse_categorical_accuracy: 0.8982
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3209 - sparse_categorical_accuracy: 0.9008


[0.3208629786968231, 0.9007999897003174]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [13]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Conv2D(
                channel_1,
                kernel_size=5,
                padding="same",
                kernel_initializer=initializer,
            ),
            tf.keras.layers.Activation("relu"),
            tf.keras.layers.Conv2D(
                channel_2,
                kernel_size=3,
                padding="same",
                kernel_initializer=initializer,
            ),
            tf.keras.layers.Activation("relu"),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(num_classes, kernel_initializer=initializer),
            tf.keras.layers.Activation("softmax"),
        ]
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model


learning_rate = 5e-4


def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate, momentum=0.9, nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer


train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.9210360050201416, Accuracy: 10.9375, Val Loss: 2.808856964111328, Val Accuracy: 14.72147274017334
Iteration 100, Epoch 1, Loss: 0.6104059219360352, Accuracy: 81.23453521728516, Val Loss: 0.3382815718650818, Val Accuracy: 89.97899627685547
Iteration 200, Epoch 1, Loss: 0.45622119307518005, Accuracy: 86.1085205078125, Val Loss: 0.27575939893722534, Val Accuracy: 91.95919799804688
Iteration 300, Epoch 1, Loss: 0.38519787788391113, Accuracy: 88.35652160644531, Val Loss: 0.2441968470811844, Val Accuracy: 92.97930145263672
Iteration 400, Epoch 1, Loss: 0.3466004729270935, Accuracy: 89.50670623779297, Val Loss: 0.2025061547756195, Val Accuracy: 94.45944213867188
Iteration 500, Epoch 1, Loss: 0.3169305920600891, Accuracy: 90.51896667480469, Val Loss: 0.18126675486564636, Val Accuracy: 94.86949157714844
Iteration 600, Epoch 1, Loss: 0.2925339341163635, Accuracy: 91.26715850830078, Val Loss: 0.16934220492839813, Val Accuracy: 95.18951416015625
Iteration 700, Epoch 1

In [14]:
model = model_init_fn()
model.compile(
    optimizer="sgd",
    loss="sparse_categorical_crossentropy",
    metrics=[tf.keras.metrics.sparse_categorical_accuracy],
)
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.2122 - sparse_categorical_accuracy: 0.9368 - val_loss: 0.1027 - val_sparse_categorical_accuracy: 0.9689
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1049 - sparse_categorical_accuracy: 0.9684


[0.104853555560112, 0.9684000015258789]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [15]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(
        hidden_size, activation="relu", kernel_initializer=initializer
    )(flattened_inputs)
    scores = tf.keras.layers.Dense(
        num_classes, activation="softmax", kernel_initializer=initializer
    )(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model


def test_two_layer_fc_functional():
    """A small unit test to exercise the TwoLayerFC model above."""
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)


test_two_layer_fc_functional()

(64, 10)


In [16]:
input_shape = (28, 28, 1)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2


def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)


def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)


train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.604059934616089, Accuracy: 7.8125, Val Loss: 3.3783118724823, Val Accuracy: 16.92169189453125
Iteration 100, Epoch 1, Loss: 0.7297687530517578, Accuracy: 77.69182586669922, Val Loss: 0.3993353843688965, Val Accuracy: 87.89878845214844
Iteration 200, Epoch 1, Loss: 0.547318160533905, Accuracy: 83.43438720703125, Val Loss: 0.3314892053604126, Val Accuracy: 90.1390151977539
Iteration 300, Epoch 1, Loss: 0.4669276773929596, Accuracy: 85.91154479980469, Val Loss: 0.3078033924102783, Val Accuracy: 90.83908081054688
Iteration 400, Epoch 1, Loss: 0.4251936972141266, Accuracy: 87.2545166015625, Val Loss: 0.27413779497146606, Val Accuracy: 91.63916015625
Iteration 500, Epoch 1, Loss: 0.3934116065502167, Accuracy: 88.29216003417969, Val Loss: 0.2581823170185089, Val Accuracy: 92.20922088623047
Iteration 600, Epoch 1, Loss: 0.36887961626052856, Accuracy: 88.98970794677734, Val Loss: 0.2580389380455017, Val Accuracy: 92.26922607421875
Iteration 700, Epoch 1, Loss: 0.34

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

---

## Эксперемент 1

В рамках данного эксперимента обучается двухблочная свёрточная нейросеть (32 и 64 фильтра с начальной инициализацией весов `HeNormal`), используется пакетная нормализация и пулинг максимумов после активации `ReLU`. Вместо традиционного выравнивания через `Flatten` применяется пространственная агрегация `GlobalAveragePooling2D`, которая выступает встроенным структурным регуляризатором и радикально уменьшает число параметров перед финальным линейным слоем. Обучение длится 10 эпох батчами по 128 объектов с использованием оптимизатора `Adam` (lr=1e-3) и функции потерь `SparseCategoricalCrossentropy`.

In [17]:
class CustomConvNet(tf.keras.Model):
    def __init__(self) -> None:
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on mnist                      #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        init = tf.keras.initializers.HeNormal()

        self.conv1 = tf.keras.layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding="same",
            kernel_initializer=init,
        )
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(
            filters=64,
            kernel_size=3,
            padding="same",
            kernel_initializer=init,
        )
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool = tf.keras.layers.MaxPooling2D(2)
        self.gap = tf.keras.layers.GlobalAveragePooling2D()
        self.fc = tf.keras.layers.Dense(10, kernel_initializer=init)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, x, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on mnist                      #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = tf.nn.relu(self.bn1(self.conv1(x), training=training))
        x = self.pool(x)
        x = tf.nn.relu(self.bn2(self.conv2(x), training=training))
        x = self.pool(x)
        x = self.gap(x)
        x = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model1 = CustomConvNet()
model1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

history1 = model1.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=10,
    validation_data=(X_val, y_val),
    verbose=1,
)

test_loss, test_acc = model1.evaluate(X_test, y_test, verbose=0)
print(f"Model 1 - Test Accuracy: {test_acc:.4f}")

Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 30ms/step - loss: 1.5228 - sparse_categorical_accuracy: 0.6056 - val_loss: 1.5455 - val_sparse_categorical_accuracy: 0.3758
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - loss: 0.8087 - sparse_categorical_accuracy: 0.8527 - val_loss: 0.7990 - val_sparse_categorical_accuracy: 0.7864
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - loss: 0.5059 - sparse_categorical_accuracy: 0.9060 - val_loss: 0.4488 - val_sparse_categorical_accuracy: 0.9034
Epoch 4/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 31ms/step - loss: 0.3654 - sparse_categorical_accuracy: 0.9271 - val_loss: 0.4530 - val_sparse_categorical_accuracy: 0.8614
Epoch 5/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - loss: 0.2883 - sparse_categorical_accuracy: 0.9404 - val_loss: 0.3250 - val_sparse_categorical_accuracy: 0.9230
Epoch 6/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - loss: 0.2431 - sparse_categorical_accuracy: 0.9471 - val_loss: 0.2895 - val_sparse_categorical_accuracy:

## Эксперемент 2

В рамках второго эксперимента исследуется трехслойная свёрточная архитектура (с последовательным увеличением числа фильтров: 16, 32 и 64) с чередованием операций `MaxPooling2D` и классическим выравниванием признаков через `Flatten` вместо глобального пулинга. Для строгого контроля переобучения здесь внедрен комплексный подход: ко всем свёрточным и финальному полносвязному слою применена L2-регуляризация (1e-4), а перед итоговым классификатором на 10 классов добавлен слой `Dropout`, случайным образом отключающий 15% нейронов. Инициализация весов сохранена по методу `HeNormal`, однако процесс оптимизации базируется на `SGD` с ускорением Нестерова (momentum=0.9, lr=1e-3).

In [18]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        init = tf.keras.initializers.HeNormal()
        reg = tf.keras.regularizers.l2(1e-4)

        self.conv1 = tf.keras.layers.Conv2D(
            16, 3, padding="same", kernel_initializer=init, kernel_regularizer=reg
        )
        self.conv2 = tf.keras.layers.Conv2D(
            32, 3, padding="same", kernel_initializer=init, kernel_regularizer=reg
        )
        self.conv3 = tf.keras.layers.Conv2D(
            64, 3, padding="same", kernel_initializer=init, kernel_regularizer=reg
        )
        self.pool = tf.keras.layers.MaxPooling2D(2)
        self.flatten = tf.keras.layers.Flatten()
        self.dropout = tf.keras.layers.Dropout(0.15)
        self.fc = tf.keras.layers.Dense(
            10, kernel_initializer=init, kernel_regularizer=reg
        )

    def call(self, x, training=False):
        x = tf.nn.relu(self.conv1(x))
        x = tf.nn.relu(self.conv2(self.pool(x)))
        x = tf.nn.relu(self.pool(self.conv3(x)))
        x = self.flatten(x)
        x = self.dropout(x, training=training)
        return self.fc(x)


model2 = CustomConvNet()
model2.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=1e-3, momentum=0.9, nesterov=True),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

history2 = model2.fit(
    X_train, y_train, batch_size=128, epochs=10, validation_data=(X_val, y_val), verbose=1
)

test_loss, test_acc = model2.evaluate(X_test, y_test, verbose=0)
print(f"Model 2 - Test Accuracy: {test_acc:.4f}")

Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.3731 - sparse_categorical_accuracy: 0.8933 - val_loss: 0.1643 - val_sparse_categorical_accuracy: 0.9586
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.1559 - sparse_categorical_accuracy: 0.9607 - val_loss: 0.1250 - val_sparse_categorical_accuracy: 0.9711
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.1212 - sparse_categorical_accuracy: 0.9705 - val_loss: 0.1038 - val_sparse_categorical_accuracy: 0.9768
Epoch 4/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.1058 - sparse_categorical_accuracy: 0.9749 - val_loss: 0.0946 - val_sparse_categorical_accuracy: 0.9788
Epoch 5/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0960 - sparse_categorical_accuracy: 0.9786 - val_loss: 0.0880 - val_sparse_categorical_accuracy: 0.9811
Epoch 6/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0869 - sparse_categorical_accuracy: 0.9807 - val_loss: 0.0839 - val_sparse_categorical_accuracy: 0.981

Опишите все эксперименты, результаты. Сделайте выводы.

## Выводы
### Эксперимент 1

Модель демонстрирует плавную, стабильную сходимость без явных признаков переобучения. На старте обучение идет относительно медленно (точность на валидации после первой эпохи составляет лишь 37,58%), однако к 10-й эпохе метрики выходят на уверенное плато: 96,33% на обучающей выборке и 95,66% на валидационной. Финальная точность на тестовых данных достигла 95,81%. Отсутствие существенного разрыва между функцией потерь на трейне (0,1551) и валидации (0,1691) подтверждает, что `GlobalAveragePooling2D` в связке с пакетной нормализацией отлично работает как структурный регуляризатор. Тем не менее, из-за сильного пространственного сжатия признаков перед финальным слоем, модели требуется больше времени на "раскачку", а ее итоговая емкость оказывается несколько ограниченной.

### Эксперимент 2

Вторая архитектура показывает лучшую скорость начального обучения и выдающуюся обобщающую способность. Уже после первой эпохи точность на валидации достигает 95,86%. В дальнейшем метрики монотонно и синхронно растут, завершая 10-ю эпоху с показателями 98,62% (трейн) и 98,48% (валидация), при итоговой точности на тесте 98,50%. Это доказывает, что комплексная стратегия контроля переобучения (L2-регуляризация + Dropout) в сочетании с оптимизатором SGD (Nesterov) позволяет сети быстро извлекать полезные признаки, не запоминая шум, и максимально полно использовать емкость полносвязного слоя после классического `Flatten`.

### Основной вывод

Сравнительный анализ метрик ясно показывает превосходство второго подхода для данной задачи. Несмотря на то что первая архитектура с `GlobalAveragePooling2D` успешно предотвращает переобучение, вторая модель (с использованием `Flatten`, явной L2-регуляризации, Dropout и классического SGD с импульсом) сходится значительно быстрее и обеспечивает прирост итоговой точности почти на 2,7% (98,50% против 95,81%). Для задач подобного типа сохранение пространственной структуры перед полносвязным слоем в комбинации с жестким контролем весов дает более высокий и стабильный результат.